# Project01 Colab Main

Colab에서 프로젝트를 실행하기 위한 정리된 노트북입니다.

실행 순서:
1. 의존성 설치
2. Git clone 후 폴더명을 `project01`로 정리
3. Google Drive, GPU, config, WandB 연결
4. 데이터 준비
5. 학습
6. 제출 이미지 zip 업로드 후 압축 해제
7. inference 실행
8. prediction zip 다운로드
9. ONNX export 및 다운로드

## 0. Colab 환경 설정 및 의존성 설치

In [ ]:
import os

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["WANDB__SERVICE_WAIT"] = "300"

!pip install -q wandb pycocotools onnx


## 1. Git clone 및 프로젝트 폴더 정리

`git clone`을 하면 `/content/skku-2-openai_pa1` 폴더가 생기므로, 이를 `/content/project01`로 바꿉니다.

처음 실행할 때는 `REPO_URL`을 본인 GitHub 주소로 바꾸세요.
이미 `/content/project01`이 있으면 clone/rename을 건너뜁니다.

In [ ]:
import os
import sys
import shutil
from pathlib import Path

REPO_URL = "https://github.com/YOUR_GITHUB_ID/skku-2-openai_pa1.git"  # TODO: 본인 repo 주소로 수정
CLONE_DIR = Path("/content/skku-2-openai_pa1")
PROJECT_ROOT = Path("/content/project01")

# project01이 없고 clone 폴더도 없으면 git clone
if not PROJECT_ROOT.exists() and not CLONE_DIR.exists():
    if "YOUR_GITHUB_ID" in REPO_URL:
        raise ValueError("REPO_URL을 본인 GitHub repository 주소로 수정한 뒤 실행하세요.")
    !git clone {REPO_URL} {CLONE_DIR}

# clone 폴더가 있고 project01이 없으면 이름 변경
if CLONE_DIR.exists() and not PROJECT_ROOT.exists():
    shutil.move(str(CLONE_DIR), str(PROJECT_ROOT))

# project01이 이미 있으면 그대로 사용
if not PROJECT_ROOT.exists():
    raise FileNotFoundError("/content/project01이 없습니다. git clone 또는 폴더 업로드 상태를 확인하세요.")

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("cwd:", os.getcwd())
print("top-level files:", sorted([p.name for p in PROJECT_ROOT.iterdir()])[:30])
print("src exists:", (PROJECT_ROOT / "src").exists())


## 2. Google Drive, GPU, config, WandB 연결

checkpoint는 Drive에 저장되도록 config의 `checkpoint.save_dir`를 확인하세요.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
import json
import torch
from pathlib import Path

from src.config.config import load_config
from src.utils.seed import set_seed
from src.utils.logger import init_wandb, get_logger

cfg = load_config("src/config/default.yaml")

logger = get_logger()

set_seed(
    cfg.runtime.seed,
    deterministic=getattr(cfg.runtime, "deterministic", False),
)

device = torch.device(cfg.runtime.device if torch.cuda.is_available() else "cpu")
print("device:", device)

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available. Colab 런타임 유형을 GPU로 바꿔주세요.")

# 중요한 경로 확인
print("checkpoint.save_dir:", cfg.checkpoint.save_dir)
print("data.root:", cfg.data.root)
print("submit.img_dir:", cfg.submit.img_dir)
print("submit.pred_dir:", cfg.submit.pred_dir)

Path(cfg.checkpoint.save_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.submit.img_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.submit.pred_dir).mkdir(parents=True, exist_ok=True)

wandb_run = init_wandb(cfg)


## 3. 데이터 준비

VOC는 `torchvision.datasets.VOCSegmentation(download=True)` 경로를 통해 코드 내부에서 받을 수 있습니다.
COCO-VOC 학습을 사용할 경우 아래 셀에서 COCO 2017 train image와 annotation을 준비합니다.

COCO를 쓰지 않을 경우, config의 `data.train_datasets`에서 `coco_voc`을 빼거나 이 셀을 건너뛰세요.

In [ ]:
from pathlib import Path
import shutil
import zipfile

COCO_ROOT = Path("/content/datasets/coco")
ANN_ZIP = COCO_ROOT / "annotations_trainval2017.zip"
IMG_ZIP = COCO_ROOT / "train2017.zip"
ANN_PATH = COCO_ROOT / "annotations" / "instances_train2017.json"
IMG_DIR = COCO_ROOT / "train2017"

COCO_ROOT.mkdir(parents=True, exist_ok=True)

def is_valid_zip(path: Path) -> bool:
    return path.exists() and zipfile.is_zipfile(path)

# 깨진 zip 방지
for zip_path in [ANN_ZIP, IMG_ZIP]:
    if zip_path.exists() and not is_valid_zip(zip_path):
        print("remove broken zip:", zip_path)
        zip_path.unlink()

# annotation 준비
if not ANN_PATH.exists():
    if not is_valid_zip(ANN_ZIP):
        !wget -c -O /content/datasets/coco/annotations_trainval2017.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    !unzip -q -o /content/datasets/coco/annotations_trainval2017.zip -d /content/datasets/coco

# image 준비
num_images = len(list(IMG_DIR.glob("*.jpg"))) if IMG_DIR.exists() else 0
if num_images == 0:
    if not is_valid_zip(IMG_ZIP):
        !wget -c -O /content/datasets/coco/train2017.zip http://images.cocodataset.org/zips/train2017.zip
    !unzip -q -o /content/datasets/coco/train2017.zip -d /content/datasets/coco

num_images = len(list(IMG_DIR.glob("*.jpg"))) if IMG_DIR.exists() else 0
print("COCO annotation:", ANN_PATH.exists(), ANN_PATH)
print("COCO images:", IMG_DIR.exists(), num_images)


## 4. Import sanity check

학습 전에 config 구조와 주요 module import가 정상인지 빠르게 확인합니다.

In [ ]:
from src.data.build import build_dataloaders, build_val_loader
from src.models.build import build_model
from src.train import main as train_main
from src.eval import main as eval_main
from src.infer import main as infer_main

print("imports ok")
print("model:", cfg.model.name, cfg.model.backbone)
print("input_size:", cfg.data.input_size)
print("num_classes:", cfg.model.num_classes)


## 5. Train

`src/train.py`는 `src/engine/trainer.py`의 `fit()`을 사용하도록 정리되어 있어야 합니다.
학습 중 `last.pth`, `best.pth`가 `cfg.checkpoint.save_dir`에 저장됩니다.

In [ ]:
from src.train import main as train_main

train_result = train_main(cfg)
print(train_result)


## 6. Evaluate best checkpoint

학습 후 `best.pth` 기준으로 validation을 한 번 더 확인합니다.

In [ ]:
from src.eval import main as eval_main

eval_result = eval_main(cfg)
print(eval_result)


## 7. 제출 이미지 폴더 생성

아래 셀은 `/content/project01/submit/img` 폴더만 만듭니다.
그 다음 사용자가 직접 zip 파일을 Colab에 업로드하면 됩니다.

추천 업로드 위치:
- `/content/project01/submit/img.zip`
- 또는 `/content/img.zip`

In [ ]:
from pathlib import Path

submit_root = Path("/content/project01/submit")
img_dir = Path(cfg.submit.img_dir)
pred_dir = Path(cfg.submit.pred_dir)

submit_root.mkdir(parents=True, exist_ok=True)
img_dir.mkdir(parents=True, exist_ok=True)
pred_dir.mkdir(parents=True, exist_ok=True)

print("submit_root:", submit_root)
print("img_dir:", img_dir)
print("pred_dir:", pred_dir)
print("여기에 zip을 업로드하세요:", submit_root / "img.zip")


## 8. 업로드한 제출 이미지 zip 압축 해제 및 정리

zip 업로드 후 실행하세요.
zip 내부 구조가 중첩되어 있어도 `.jpg`, `.jpeg`, `.png` 이미지를 찾아서 `cfg.submit.img_dir`로 모읍니다.

In [ ]:
from pathlib import Path
import shutil
import zipfile

submit_root = Path("/content/project01/submit")
img_dir = Path(cfg.submit.img_dir)

ZIP_CANDIDATES = [
    submit_root / "img.zip",
    submit_root / "submit.zip",
    Path("/content/img.zip"),
    Path("/content/submit.zip"),
]

zip_path = next((p for p in ZIP_CANDIDATES if p.exists()), None)
if zip_path is None:
    raise FileNotFoundError(
        "제출 이미지 zip을 찾지 못했습니다. "
        f"다음 위치 중 하나에 업로드하세요: {ZIP_CANDIDATES}"
    )

extract_dir = submit_root / "_uploaded_images"
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True, exist_ok=True)

# 기존 이미지 제거 후 새로 정리
for p in img_dir.glob("*"):
    if p.is_file():
        p.unlink()

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

image_exts = {".jpg", ".jpeg", ".png"}
image_files = sorted([p for p in extract_dir.rglob("*") if p.suffix.lower() in image_exts])

if len(image_files) == 0:
    raise RuntimeError(f"zip 안에서 이미지 파일을 찾지 못했습니다: {zip_path}")

for src in image_files:
    dst = img_dir / src.name
    # 파일명이 중복되면 stem_001 형태로 저장
    if dst.exists():
        stem, suffix = src.stem, src.suffix
        k = 1
        while True:
            candidate = img_dir / f"{stem}_{k:03d}{suffix}"
            if not candidate.exists():
                dst = candidate
                break
            k += 1
    shutil.copy2(src, dst)

print("zip_path:", zip_path)
print("extracted images:", len(image_files))
print("prepared images:", len(list(img_dir.glob('*'))))
print("img_dir:", img_dir)


## 9. Inference

`cfg.submit.img_dir`의 이미지들을 읽어서 `cfg.submit.pred_dir`에 prediction mask를 저장합니다.

In [ ]:
from src.infer import main as infer_main

infer_main(cfg)


## 10. Prediction zip 생성 및 다운로드

In [ ]:
from pathlib import Path
import shutil
from google.colab import files

pred_dir = Path(cfg.submit.pred_dir)
zip_base = Path("/content/project01/submit/pred")
zip_path = Path(str(zip_base) + ".zip")

if zip_path.exists():
    zip_path.unlink()

if not pred_dir.exists() or len(list(pred_dir.glob("*.png"))) == 0:
    raise RuntimeError(f"prediction png가 없습니다: {pred_dir}")

shutil.make_archive(str(zip_base), "zip", root_dir=pred_dir)
print("created:", zip_path)
print("num pred png:", len(list(pred_dir.glob("*.png"))))

files.download(str(zip_path))


## 11. ONNX export 및 다운로드

`best.pth`를 불러와 ONNX 파일을 생성합니다.
과제 제출에서 ONNX가 필요하면 이 셀까지 실행하세요.

In [ ]:
from pathlib import Path
import os
import torch
import onnx
from google.colab import files

from src.models.build import build_model

ckpt_path = getattr(cfg.checkpoint, "resume_path", "")
if ckpt_path is None or ckpt_path == "":
    ckpt_path = os.path.join(cfg.checkpoint.save_dir, "best.pth")

ckpt_path = Path(ckpt_path)
if not ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

model = build_model(cfg).to(device)
checkpoint = torch.load(ckpt_path, map_location=device)

if "model" in checkpoint:
    model.load_state_dict(checkpoint["model"])
elif "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.eval()

onnx_path = Path("/content/project01/submit/model.onnx")
onnx_path.parent.mkdir(parents=True, exist_ok=True)

input_size = int(cfg.data.input_size)
dummy = torch.randn(1, 3, input_size, input_size, device=device)

with torch.no_grad():
    torch.onnx.export(
        model,
        dummy,
        str(onnx_path),
        input_names=["input"],
        output_names=["logits"],
        opset_version=17,
        do_constant_folding=True,
        dynamic_axes={
            "input": {0: "batch", 2: "height", 3: "width"},
            "logits": {0: "batch", 2: "height", 3: "width"},
        },
    )

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)

print("ONNX saved:", onnx_path)
print("ONNX size MB:", onnx_path.stat().st_size / 1024 / 1024)

files.download(str(onnx_path))


## 12. 선택: 현재 git 상태 확인

Colab에서 수정한 파일이 있다면 push 전에 확인하세요.

In [ ]:
%cd /content/project01
!git status --short
